# Artifical Intelligence Project
CC2006: Inteligência Artificial  
Guilherme Rodrigues  -  202405814  
Lara Dinis  -  202406361  
Miguel Ribeiro  -  202405276  
Maio 2026


This Notebook documents de development of a variant of the classical Connect Four game, where a player can also Pop their previous moves at the bottom of a collumn. Our implementations consists of three versions of MCTS, each progressively more efficient, as well as a Decision Tree built with the ID3 algorithm and trained with the Iris Dataset.

## Requirements and Setup

In order to run all files that compose the project it is needed, at least:
- Pyhton 3.10
- Pygame 2.5.0
- Numpy 1.24.0
- Numba 0.57.0
- Matplotlib 3.7.0
- Networkx 3.0
- Pyvis 0.3.0
- Pandas 2.0.0, or more recent versions.

## Index

1. PopOut
2. Project Structure
3. Monte Carlo Tree Search
4. MCTS's Implementations
5. BitBoard Representation
6. Numba Acceleration
7. ID3 Decision Tree
8. Game Architecture
9. Results and Performance Comparison
10. Discussion
11. References


## 1. PopOut

### Complete Game Rules

PopOut is a variant of Connect-4. It starts the same as traditional gameplay, with an empty board
and players alternating turns placing their own coloured pieces into the board. 

During each turn, a player can either add another piece from the top, or if one has any discs of their own colour on the bottom row, remove (or “pop out”).

The first player to connect four of their pieces horizontally, vertically, or diagonally wins the game.

If the board is full or same board state appears 3 times, the players can choose to draw, ending the game.

In the case of a pop move leading to both players connecting four, the player who made the move is declared the winner of the match.

## 2. Project Structure

| File | Function |
|----------|--------|
| `game_logic.py` | Basic game logic: game flow functions using matrices |
| `bitboard.py` | Bitboard representation of the gameboard: game represented with bit-wise operations |
| `MCTS.py` | Original version of MCTS with rollout heiristics (EASY mode) |
| `MCTS_fast.py` | MCTS optimization: without heuristics, board as a plane tuple (MEDIUM mode) |
| `MCTS_bitboard.py` | MCTS with bitboard + rollout compiled in C with Numba (HARD mode) |
| `ID3.py` | Implementação do algoritmo ID3 para árvores de decisão |
| `iris_loader.py` | Loading and Discretization of the Iris Dataset for ID3 testing |
| `main_pygame.py` | Graphical Pygame Interface, menus and main game loop |
| `dataset.csv` | Dataset generated by several simulations of MCTS playouts to train the ID3 |

## 3. Monte Carlo Tree Search

Monte Carlo Tree Search is a search algorithm that progressively builds a decision tree through random simulations, learning the value of each possible move through multiple random rollouts.

### The 4 phases

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────────┐
│  SELECTION  │ →  │  EXPANSION  │ →  │  SIMULATION │ →  │ RETROPROPAGATION│
│             │    │             │    │  (rollout)  │    │                 │
│ Goes down   │    │ Creates a   │    │ Plays       │    │ Updates wins    │
│ the tree,   │    │ new child   │    │ randomly    │    │ and visits      │
│ guided by   │    │ node (unseen│    │ until it    │    │ of every node   │
│ UCT         │    │ movement)   │    │ reaches a   │    │ in the path     │
│             │    │             │    │ terminal    │    │                 │
│             │    │             │    │ state       │    │                 │
└─────────────┘    └─────────────┘    └─────────────┘    └─────────────────┘
```

### UCT — Upper Confidence Bound for Trees

The UCT formula balances exploration (visit less frequently visited nodes) and exploitation (visit nodes with good win rates):

$$UCT = \frac{wins}{visits} + C \cdot \sqrt{\frac{\ln(N)}{visits}}$$

Where:
- `wins/visits` → Node win rate (exploitation)
- `C * sqrt(ln(N)/visits)` → Bonus for less frequently visited nodes (exploration)
- `C = 1.41 ≈ √2` → Exploration constant

### Tree Reuse

After each move, instead of rebuilding the tree in its entirety, the subtree corresponding to the chosen move is reused, becoming the new root and mantaining all the accumulated information.

## 4. MCTS's Implementations

### MCTS.py - EASY mode

- Game board represented as a matrix
- A copy of the board is created at each node
- Rollout with heuristics: checks immediate victories and blocking moves at every step
- Main problem: for each rollout simulation, the heursitic functions result in overly complex move evaluation, with only about approximately 600 iterations per second.

```python
for m in moves:
    info = apply_move(temp_board, curr, m)
    if gl.check_victory(temp_board, curr):  # verifica vitória
        best_move = m; break
    undo_move(temp_board, info)
    info_op = apply_move(temp_board, 3 - curr, m)
    if gl.check_victory(temp_board, 3 - curr):  # verifica bloqueio
        best_move = m
    undo_move(temp_board, info_op)
```

---

### MCTS_fast.py - MEDIUM mode

1. Purely random rollouts: removes all heuristics, converging to the best decisions with a high enough volume of iterations, trading complexity with experience.

2. Board represented as a tuple (every cell as a number), which are immutable, avoiding the necessity of creating copies.

```python
# Antes: cópia cara em cada nó
self.board = [row[:] for row in board]  # aloca 6 listas novas

# Depois: sem cópia
self.board = board  # tuplo imutável, partilha referência
```

3. Moves as integers, instead of tuples (for example, `("drop", 3)`), eliminating tuple allocation.

4. Game operations return a new tuple instead of in-place modifications.

```python
def drop_tuple(board, peca, col):
    lst = list(board)      # cópia temporária (42 inteiros)
    lst[idx] = peca        # modifica
    return tuple(lst)      # devolve novo tuplo imutável
```

As a result, this new implementation can generate up to 8 000 iterations per second, over x13 more iterations than the simpler version of the algorithm.

---

### MCTS_bitboard.py — HARD mode

1. Bitboard representation: the board becomes two 64-bit integers, each representing a cell of the board.

2. `__slots__` in class Node reduces the time spent in memory per object:
```python
__slots__ = ('p1', 'p2', 'jogador_atual', 'move', 'parent',
             'children', 'visits', 'wins', 'untried_moves')
```

3. Fixed time instead of fixed iterations: in earlier stages of the game performs less iterations, increasing them throughout the later stages, improving the algorithm's time efficiency.

4. Rollout compiled with Numba, in native machine code.

As a result, this implementation can achieve about 140 000 iterations per second at the start of the game and 3 000 000 iterations per second in the later stages, much higher than both previous implementations.


## 5. BitBoard Representation

### Concept

Instead of saving the game board as a matrix, two 64-bit integers are used, each representing a player, where each bit represents a cell in the original board.

To add a piece to the board, we must shift the corresponding bit in the current player's integer following the formula `row * 7 + collumn`

### Check Victory in Bitboard

To verify if a terminal state has been reached in this implementation it is only needed 2 bit-wise operations, instead of the over 60 comparisons used in the simpler version.

```python
# Horizontal (shift de 1)
m = p & (p >> 1)
if m & (m >> 2): return True  # 4 em linha!

# Vertical (shift de 7)
m = p & (p >> 7)
if m & (m >> 14): return True
```

How it works:
- `p >> 1` shits all pieces one collumn to the left
- `p & (p >> 1)` finds consecutive pairs
- `m & (m >> 2)` finds two pairs with 2 positions of distanceencontra dois pares com 2 posições de distância, idem 4 in line

### Horizontal Mask

To avoid pieces in the end of a line to be considered consecutive with pieces in the beggining of the next one a mask is used.

```python
MASK_HORIZONTAL = 0
for r in range(6):
    MASK_HORIZONTAL |= 0b0111111 << (r * 7)  # apaga coluna 6 de cada linha

m = (p & MASK_HORIZONTAL) & ((p & MASK_HORIZONTAL) >> 1)
```


## 6. Numba Acceleration

### What is Numba

Numba is a Python library which compiles Python fucntions to native machine code using JIT (Just-In-Time compilation). The first call complies the function and the next are executed at the velocity of C.

### Why only `simulate`

Numba works best with simple lopps over integers, arithmetic and bit-wise operations, and fixed-size lists or numpy arrays. However, it does not work with arbitrary Pyhton objects, dinamic lists and module calls. Therefore, only `simulate` (rollout) is compiled with Numba, called once per iteration.

### Adaptations

Numba does not support `list.append()`, so the dynamic list had to be substituted by a fixed-size array.

`check_victory` is also inline inside of simulate, with external calls:

```python
@njit
def simulate(p1, p2, jogador_atual):
    curr = jogador_atual
    for _ in range(42):
        # ... rollout ...
        
        # check_victory inline (operações de bits puras)
        m = p1 & (p1 >> 1)
        if m & (m >> 2): return 1
        m = p2 & (p2 >> 1)
        if m & (m >> 2): return 2
        # ... vertical, diagonais ...
    return 0
```

### Impact

| Game Stage | Before Numba | After Numba |
|-------------|----------------|-----------------|
| Start | ~50k iter/move | ~140k iter/move |
| End | ~300k iter/move | ~3M iter/move |


## 7. ID3 Decision Tree

### Conceito

The ID3 (Iterative Dichotomiser 3) algorithm builds a Decision Tree choosing at each node the attribute that maximized the information gain.

### Entropy and Information Gain

Entropy measures the impurity of a given set of examples:

$$H(S) = -\sum_{c} p_c \log_2(p_c)$$

Information Gain of an attribute A:

$$IG(S, A) = H(S) - \sum_{v} \frac{|S_v|}{|S|} H(S_v)$$

The attribute with the highest Information Gain is chosen for the split.

### Connect Four Apliccation

The dataset was generated by several MCTS vs MCTS playouts:
- Attributes: `cell_0_0` to `cell_5_6`, the value of each cell in the board
- Label: the move chosen by the MCTS algorithm (`drop_3`, `pop_1`, etc.)

The tree learns the better playing patterns of MCTS and becomes able to replicate them without simulations, which is far more faster but less precise.

### Limitations

The Decision Tree is deterministic and can only answer depending on the learnt patterns. Against unpredictabilty falls back on randomness as the preferred decision.

### Visualization

The decision tree can be visualized in two different ways:
```python
tree.plot_tree("decision_tree.png")         # PNG estático
tree.plot_interactive("arvore_interativa.html")  # HTML interativo com zoom
```


## 8. Game Architecture

### Game Modes

| Mode | Description |
|------|-----------|
| Human vs Human | Two human players |
| Human vs AI | Human chooses the AI model |
| AI vs AI | Each player chooses its agent independently |

### Agent Selection

As seen, there are 4 agents that can be chosen:

| Difficulty/Type | File | Configuration |
|----------------|------|---------------|
| MCTS Fácil | MCTS.py | 6000 fixed iterations, with heuristics |
| MCTS Médio | MCTS_fast.py | 1 second to think, tuples |
| MCTS Difícil | MCTS_bitboard.py | 1 second to think, bitboard + Numba |
| Decision Tree | ID3.py | instantaneous, learnt patterns |

### Tree Reuse in AI vs AI

Each player mantains its own root:
```python
mcts_root_p1 = None  # root do jogador 1
mcts_root_p2 = None  # root do jogador 2
```

After each move, the adversary's root is updated, the subtree corresponding to the opponent's move becomes the new root, avoiding the total reconstruction of the tree.

### Tie Rules

- State Repetition: if the same board state is seen 3 times, either player can declare the game drawn
- Full Board: if the board is completely full and there is not a winner, the current player can choose to end the game or perform a pop move
- Apparent tie: if a pop move results in both players connecting four pieces, the player who played last (the pop) is declared the winner

In Human vs Human mode appears a button to declare tie. In AI vs AI mode, the tie is automatic.


## 9. Results and Performance Comparison

### Iterations per Seconds

| Version | Start of the Game | End of the Game | Applied Methods |
|--------|---------------|-------------|-------------------|
| MCTS | ~600 | ~600 | Rollout with heuristics |
| MCTS_fast | ~8 000 | ~8 000 | Pure rollout, tuples, integers |
| MCTS_bitboard (without Numba) | ~28 000 | ~100 000 | Bitboard, slots, fixed time |
| MCTS_bitboard (with Numba) | ~140 000 | ~600 000 | Rollout compiled with JIT |

### Iterations per Move (5 seconds)

| Game Phase | MCTS_bitboard + Numba |
|-------------|----------------------|
| Move 1 | ~140 000 |
| Move 5 | ~150 000 |
| Move 10 | ~160 000 |
| Move 20 | ~300 000 |
| Move 30 | ~3 000 000 |

Each rollout either fills the board until the end or until one player wins. With a clean board each rollout takes between 20 and 40 steps. However, with an almost full board it takes up to 3 steps. Taking this into consideration, the same second of time produces many more iterations.

The use of fixed time instead of iterations takes advantage of this phenomenom: when the decisions are more critical, the MCTS is given more resources to enact the best decision.

### Optimizations Summary

```
MCTS 
    │ ×13  Remove heuristics + tuples + moves as integers
    ▼
MCTS_fast
    │ ×3   Bitboard + __slots__ + fixed time
    ▼
MCTS_bitboard (without Numba)
    │ ×5   Rollout compiled with JIT
    ▼
MCTS_bitboard (with Numba)
    
Total: ×200 compared to the original
```


## 10. Discussion

### Hyperparameter Tuning

ITERACOES_EASY = 6000: balances speed and quality. At 600 iterations/second, this takes ~10 seconds per move - acceptable wait time for casual play. Beyond 6000, diminishing returns make further increases inefficient.

POP_BIAS = 0.4: POP moves occur ~20-30% of the time in real play. Setting bias to 0.4 slightly favors DROPs but gives POPs realistic representation.

C = 1.414 (√2): extensively validated across Go, Chess, and Connect Four. Lower values cause premature exploitation; higher values waste computation on exploration.

max_depth = None (unlimited): Connect Four positions are highly complex with 42 cells × 3 values. Limiting depth cuts accuracy from ~65% to ~45% in testing. Full depth captures all spatial patterns needed for strong play.

TEMPO = 5.0 seconds: maximum acceptable wait time for human players. Dynamically allocates computation where it matters most - early game (fewer iterations) vs critical endgame (millions of iterations). Values below 3 seconds weaken play; above 10 seconds offer minimal improvement.

### Why MCTS Works for Connect Four
Connect Four has several properties that make MCTS effective. The game is deterministic, meaning there is no randomness in gameplay. It is a perfect information game where both players see everything on the board. The branching factor is moderate, with approximately five to eight moves available per position on average. The game also has clear terminal states where win, loss, or draw conditions are well-defined.

### When Decision Trees Fail
Decision trees struggle with several important scenarios. They cannot handle novel positions effectively because they can only respond to patterns that were present in their training data. They lack the ability to see ahead or plan sequences of moves like MCTS can. Additionally, decision trees are exploitable by consistent opponents who learn their predictable patterns and play accordingly.

### Bitboard Limitations
Despite their performance advantages, bitboards have several limitations. They are considerably harder to debug than simple matrix representations because inspecting individual bits requires conversion. Bit operation behavior can vary across different CPU architectures, potentially leading to portability issues.

### POP Move Challenges
The POP variant introduces several unique challenges to the game. Cyclic positions can now occur since pieces can be removed, necessitating repetition detection to avoid infinite games. The complexity increases significantly because the branching factor grows by approximately sixty percent compared to standard Connect Four.

## 11. References

- [Connect 4 Solved Position Database](https://tromp.github.io/c4/c4.html)
- [MCTS Python Tutorial](http://mcts.ai/pythonexample.html)
- [Bitboard Tutorial](https://www.chessprogramming.org/Bitboards)

### Code Repositories

- Project repository: https://github.com/MiguelFilipe12/POPOUT
- Pygame documentation: [pygame.org/docs](https://www.pygame.org/docs/)